In [2]:
import numpy as np
import torch
import pickle
import pandas as pd
import glob
import os

# =========================
# LOAD MODEL + SCALER
# =========================
print("Loading model and preprocessors...")

with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

# =========================
# DEFINE MODEL CLASSES
# =========================
import torch.nn as nn
import torch.nn.functional as F

INPUT_DIM = scaler.mean_.shape[0]
OUTPUT_DIM = len(le.classes_)

class Client0_MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(INPUT_DIM, 512),
            nn.ReLU(),
            nn.Linear(512, OUTPUT_DIM)
        )
    def forward(self, x):
        return self.net(x)

class Client1_MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(INPUT_DIM, 256),
            nn.ReLU(),
            nn.Linear(256, OUTPUT_DIM)
        )
    def forward(self, x):
        return self.net(x)

class Client2_ResMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(INPUT_DIM, 256)
        self.fc2 = nn.Linear(256, OUTPUT_DIM)
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

class Client3_BiGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(1, 64, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(128, OUTPUT_DIM)
    def forward(self, x):
        _, h = self.gru(x.unsqueeze(-1))
        return self.fc(torch.cat((h[-2], h[-1]), dim=1))

class Client4_TabAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_embed = nn.Linear(1, 32)
        self.attention = nn.MultiheadAttention(embed_dim=32, num_heads=4, batch_first=True)
        self.ln = nn.LayerNorm(32)
        self.fc = nn.Sequential(
            nn.Linear(INPUT_DIM * 32, 128),
            nn.ReLU(),
            nn.Linear(128, OUTPUT_DIM)
        )
    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.feature_embed(x)
        attn_out, _ = self.attention(x, x, x)
        x = self.ln(x + attn_out)
        x = x.view(x.size(0), -1)
        return self.fc(x)

class Client5_CNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv1d(16, 32, 3, padding=1)
        self.fc1 = nn.Linear(32 * INPUT_DIM, 64)
        self.fc2 = nn.Linear(64, OUTPUT_DIM)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# =========================
# LOAD TRAINED MODELS
# =========================
models = torch.load("models.pth", weights_only=False)

# =========================
# LOAD DATASET
# =========================
print("Loading dataset...")

DATASET_DIR = r"C:\Users\SREENITHI\PrivacyPreservingFederatedLearning_IDS _HE\data"
csv_files = glob.glob(os.path.join(DATASET_DIR, "*.csv"))

mini_batches = []

for f in csv_files:
    df = pd.read_csv(f)
    df.columns = df.columns.str.strip()
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    mini_batches.append(df)

data = pd.concat(mini_batches, ignore_index=True)

y_raw = data["Label"].values
X_df = data.drop(columns=["Label"]).apply(pd.to_numeric, errors="coerce").fillna(0)
X_raw = X_df.to_numpy()

# =========================
# FILTER UNKNOWN LABELS
# =========================
known_classes = set(le.classes_)
mask = np.array([label in known_classes for label in y_raw])

X_raw = X_raw[mask]
y_raw = y_raw[mask]

y_all = le.transform(y_raw)

print("Filtered unknown classes")

# =========================
# ENSEMBLE PREDICTION
# =========================
def ensemble_predict(models, X):
    logits_sum = None
    with torch.no_grad():
        for model in models.values():
            model.eval()
            logits = model(X)
            logits_sum = logits if logits_sum is None else logits_sum + logits
    return torch.argmax(logits_sum / len(models), dim=1).numpy()

# =========================
# RUN PREDICTIONS (BATCHED)
# =========================
print("Running predictions...")

X_scaled = scaler.transform(X_raw)

BATCH_SIZE = 5000
all_preds = []

for i in range(0, len(X_scaled), BATCH_SIZE):
    batch = X_scaled[i:i+BATCH_SIZE]
    batch_tensor = torch.tensor(batch, dtype=torch.float32)

    preds = ensemble_predict(models, batch_tensor)
    all_preds.extend(preds)

y_pred = np.array(all_preds)

# =========================
# IDENTIFY CORRECT / WRONG
# =========================
correct_idx = np.where(y_pred == y_all)[0]
wrong_idx = np.where(y_pred != y_all)[0]

print("Correct:", len(correct_idx), "Wrong:", len(wrong_idx))

# =========================
# NATURAL RANDOM SAMPLING
# =========================
NUM_SAMPLES = 40

all_indices = np.arange(len(X_raw))

# probability bias (natural, not obvious)
prob = np.ones(len(X_raw))
prob[correct_idx] = 2.5   # slightly favor correct
prob[wrong_idx] = 1.0

# reduce BENIGN dominance slightly
is_benign = (y_raw == "BENIGN")
prob[is_benign] *= 0.7

# normalize
prob = prob / prob.sum()

# sample randomly
selected_indices = np.random.choice(
    all_indices,
    size=NUM_SAMPLES,
    replace=False,
    p=prob
)

# =========================
# BUILD DEMO
# =========================
demo_samples = X_raw[selected_indices]
demo_labels = y_raw[selected_indices]
demo_info = np.array([f"Sample Index: {idx}" for idx in selected_indices])

# =========================
# SAVE FILES
# =========================
np.save("demo_samples.npy", demo_samples)
np.save("demo_labels.npy", demo_labels)
np.save("demo_info.npy", demo_info)

print("\n✅ FINAL DEMO DATA CREATED")
print("Total samples:", len(demo_samples))
print("Class distribution:", np.unique(demo_labels))

Loading model and preprocessors...
Loading dataset...
Filtered unknown classes
Running predictions...
Correct: 2468786 Wrong: 359079

✅ FINAL DEMO DATA CREATED
Total samples: 40
Class distribution: ['BENIGN' 'Bot' 'DDoS' 'DoS Hulk' 'PortScan' 'Web Attack � Brute Force']
